In [ ]:
import logging
import gradio as gr
from agents.planning_agent import PlanningAgent, VERDICT_EMOJI

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s", datefmt="%H:%M:%S")

_agent = None

def _get_agent():
    """Return (or lazily initialise) the singleton PlanningAgent."""
    global _agent
    if _agent is None:
        _agent = PlanningAgent()
    return _agent

_VERDICT_COLORS = {
    "Great Deal": "#16a34a",
    "Good Deal":  "#2563eb",
    "Fair Price": "#d97706",
    "Overpriced": "#dc2626",
    "Unknown":    "#6b7280",
}

def _product_card(p: dict) -> str:
    """
    Render a single product as a self-contained HTML card.

    Args:
        p: Enriched product dict from PlanningAgent.

    Returns:
        HTML string for one product card.
    """
    scraped   = float(p.get("price", 0))
    estimated = float(p.get("estimated_price", 0))
    savings   = float(p.get("savings_pct", 0))
    verdict   = p.get("verdict", "Unknown")
    color     = _VERDICT_COLORS.get(verdict, "#6b7280")
    emoji     = VERDICT_EMOJI.get(verdict, "❓")
    title     = p.get("title", "")
    brand     = p.get("brand", "")
    category  = p.get("category", "")
    desc      = p.get("description", "")

    savings_html = (
        f'<span style="color:#dc2626;font-size:0.8em">▲ {abs(savings):.1f}% over est.</span>'
        if savings < 0
        else f'<span style="color:#16a34a;font-size:0.8em">▼ {savings:.1f}% below est.</span>'
    )

    return f"""
    <div style="
        background:#fff;
        border:1px solid #e2e8f0;
        border-left:4px solid {color};
        border-radius:10px;
        padding:16px 20px;
        margin-bottom:12px;
        font-family:system-ui,sans-serif;
    ">
        <div style="display:flex;justify-content:space-between;align-items:flex-start;flex-wrap:wrap;gap:8px;">
            <div style="flex:1;min-width:0;">
                <div style="font-weight:700;font-size:0.95em;color:#1e293b;margin-bottom:4px;line-height:1.4">{title}</div>
                <div style="color:#94a3b8;font-size:0.78em;margin-bottom:6px">{brand} · {category}</div>
                <div style="color:#475569;font-size:0.83em">{desc}</div>
            </div>
            <div style="text-align:right;white-space:nowrap;min-width:130px;">
                <div style="
                    display:inline-block;
                    background:{color};
                    color:#fff;
                    padding:4px 12px;
                    border-radius:20px;
                    font-weight:700;
                    font-size:0.82em;
                    margin-bottom:8px;
                ">{emoji} {verdict}</div>
                <div style="font-size:1.1em;font-weight:700;color:#1e293b">${scraped:.0f}</div>
                <div style="font-size:0.78em;color:#94a3b8">est. ${estimated:.2f}</div>
                <div>{savings_html}</div>
            </div>
        </div>
    </div>
    """

def run_search(query: str):
    """
    Run the PlanningAgent pipeline and return rendered product cards.

    Args:
        query: User's natural-language product search.

    Returns:
        HTML string of product cards.
    """
    if not query.strip():
        return "<p style='color:#dc2626;font-family:system-ui'>Please enter a search query.</p>"
    try:
        enriched, _ = _get_agent().run(query)
    except Exception as exc:
        return f"<p style='color:#dc2626;font-family:system-ui'><b>Error:</b> {exc}</p>"

    cards = "".join(_product_card(p) for p in enriched)
    legend = """
    <div style="font-family:system-ui;font-size:0.78em;color:#94a3b8;margin-top:4px;padding:0 2px">
        🔥 Great Deal &nbsp;·&nbsp; ✅ Good Deal &nbsp;·&nbsp; 🟡 Fair Price &nbsp;·&nbsp; ❌ Overpriced
    </div>
    """
    return legend + cards

# ── Layout ──────────────────────────────────────────────────────────────────

with gr.Blocks(title="AI Deal Finder") as demo:

    gr.HTML("""
    <div style="font-family:system-ui,sans-serif;padding:24px 8px 8px">
        <h1 style="margin:0;font-size:1.6em;color:#1e293b">🛍️ AI Deal Finder</h1>
        <p style="margin:4px 0 0;color:#64748b;font-size:0.9em">
            Find the best deals on Jumia Kenya — powered by a fine-tuned Llama pricing model.
        </p>
    </div>
    """)

    query_input = gr.Textbox(
        label="",
        placeholder='e.g. "budget phones", "laptops for students", "wireless earbuds"',
        lines=1,
        show_label=False,
    )
    search_btn = gr.Button("Find Deals 🔍", variant="primary")

    results_html = gr.HTML()

    search_btn.click(fn=run_search, inputs=[query_input], outputs=[results_html])
    query_input.submit(fn=run_search, inputs=[query_input], outputs=[results_html])

demo.launch(share=True)
